# Chapter 08: Applications — Cryptography (Shor's Algorithm)

Welcome to **Section 1.3: General Lecture on Quantum Technology**.

This is the algorithm that started it all. The "Killer App".

In 1994, Peter Shor proved that a quantum computer could factor large integers exponentially faster than any classical computer.

Why do we care about factoring numbers? Because the security of the entire internet (**RSA Encryption**) relies on the fact that multiplying two primes is easy, but finding them back is hard.

If you can run Shor's Algorithm, you can break RSA. You can read encrypted emails, forge bank signatures, and potentially access nuclear launch codes.

In this notebook, we look at how it works.

---

## Part 1: The Problem (RSA)

**The Lock:**
Public Key ($N$) = $p \times q$. 
Example: $15 = 3 \times 5$.

**The Key:**
The prime factors $p$ and $q$.

If $N$ is small (15), finding 3 and 5 is trivial. 
If $N$ is huge (2048 bits, or 617 digits), finding $p$ and $q$ would take a classical supercomputer billions of years.

This asymmetry is the foundation of public-key cryptography.

---

## Part 2: The Quantum Insight (Period Finding)

Peter Shor realized something brilliant. Factoring is secretly a **Period Finding** problem.

Consider the function:
$$ f(x) = a^x \mod N $$

Pick a random number $a < N$. Calculate the sequence:
$a^1, a^2, a^3, ... \pmod N$

Example: $N=15, a=7$.
1. $7^1 \mod 15 = 7$
2. $7^2 \mod 15 = 49 \to 4$
3. $7^3 \mod 15 = 343 \to 13$
4. $7^4 \mod 15 = 2401 \to 1$
5. $7^5 \mod 15 = 16807 \to 7$ (Repeats!)

The sequence is: $7, 4, 13, 1, 7, 4, 13, 1...$
The **Period** ($r$) is 4. It repeats every 4 steps.

**Shor's Magic Trick:**
If you find the period $r$, there is a high probability that the greatest common divisor of $(a^{r/2} \pm 1, N)$ gives you the factors $p$ and $q$.

Let's test it:
$r = 4$. $r/2 = 2$.
$a^{r/2} - 1 = 7^2 - 1 = 48$.
$a^{r/2} + 1 = 7^2 + 1 = 50$.

$\gcd(48, 15) = 3$. (Factor found!)
$\gcd(50, 15) = 5$. (Factor found!)

boom. We broke RSA.

In [ ]:
import math

# Classical Period Findng (Slow)
def find_period_classical(a, N):
    r = 1
    t = a
    while t != 1:
        t = (t * a) % N
        r += 1
    return r

N = 15
a = 7
period = find_period_classical(a, N)
print(f"The period of f(x) = {a}^x mod {N} is: {period}")

# Classical Factoring using period
if period % 2 != 0:
    print("Period is odd, method failed.")
else:
    x = a**(period//2)
    factor1 = math.gcd(x - 1, N)
    factor2 = math.gcd(x + 1, N)
    print(f"Factors found: {factor1}, {factor2}")

---

## Part 3: Why Do We Need a Quantum Computer?

Finding the period classically is hard. You have to check $x=1, 2, 3...$ until it repeats. The period $r$ can be as large as $N$. If $N$ has 600 digits, good luck.

**Quantum Approach:**
1. Create a massive superposition of all inputs $|x\rangle$.
2. Compute $f(x) = a^x \mod N$ in superposition.
3. Use the **Quantum Fourier Transform (QFT)**.

The QFT detects the "frequency" of the function. Just like valid radio waves interfere constructively and static cancels out, the QFT amplifies the state corresponding to the period $r$.

We measure, we get $r$, we calculate the factors.

This reduces the time from Exponential ($2^N$) to Polynomial ($N^3$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualizing Periodicity
# f(x) = 7^x mod 15

x_vals = np.arange(0, 20)
y_vals = [pow(7, int(x), 15) for x in x_vals]

plt.figure(figsize=(12, 5))
plt.step(x_vals, y_vals, where='mid', color='purple', linewidth=2)
plt.plot(x_vals, y_vals, 'o', color='purple')

plt.title(f"Periodic Function: $7^x \mod 15$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True, alpha=0.3)

# Highlight the repeating pattern
for i in range(0, 20, 4):
    plt.axvline(i, color='red', linestyle='--', alpha=0.3)

plt.text(2, 10, "Period r=4", color='red', fontsize=12, ha='center')
plt.show()

---

## Conclusion: The Clock is Ticking

Shor's algorithm is mathematically proven. The only thing stopping us from breaking RSA today is hardware.

To factor a 2048-bit number, we need about **20 million physical qubits** (because of the need for error correction). We currently have ~1000 noisy ones.

Estimates say we might have a cryptographically relevant quantum computer by 2035 or 2040. This is why the world is moving to **Post-Quantum Cryptography** (Lattice-based encryption) now.

**Navigation:** [Next → Chapter 09: Applications (Material Science)](09_applications_materials.ipynb)